# 02 Asset Data Cleaning

## Goal

This notebook creates the clean asset dataset used for the climate exposure-screening project.

The raw REPD dataset contains 13,995 project records and 53 fields. Not all fields are needed for this first version of the project.

This notebook focuses on:

1. selecting the fields needed to identify, describe and locate assets,
2. cleaning the `Development Status` field,
3. filtering to operational assets,
4. removing assets without valid coordinates,
5. saving a cleaned dataset for later geospatial analysis.

The output of this notebook is a cleaned table of operational renewable energy assets with valid coordinate data.

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

DATA_INTERIM.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

csv_files = list(DATA_RAW.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {DATA_RAW}")

raw_asset_file = csv_files[0]

print("Using:", raw_asset_file.name)

df_raw = pd.read_csv(raw_asset_file, encoding="latin1")

print("Raw rows:", df_raw.shape[0])
print("Raw columns:", df_raw.shape[1])

Using: repd_publication_q4_2025_raw.csv
Raw rows: 13995
Raw columns: 53


## Selecting useful fields

The raw dataset includes project identifiers, planning history, technology information, capacity, location fields and administrative geography.

For this Version 1 exposure-screening model, I keep only the fields needed to answer four questions:

1. What is the asset?
2. What type of technology is it?
3. Is it operational?
4. Where is it located?

I also keep installed capacity because it can be used as a simple proxy for asset scale or infrastructure importance.

In [2]:
useful_columns = [
    "Ref ID",
    "Operator (or Applicant)",
    "Site Name",
    "Technology Type",
    "Installed Capacity (MWelec)",
    "Development Status",
    "Development Status (short)",
    "County",
    "Region",
    "Country",
    "Post Code",
    "X-coordinate",
    "Y-coordinate",
    "Planning Authority",
    "Under Construction",
    "Operational",
    "Solar Site Area (sqm)",
]

assets = df_raw[useful_columns].copy()

print("Selected rows:", assets.shape[0])
print("Selected columns:", assets.shape[1])

assets.head()

Selected rows: 13995
Selected columns: 17


,Ref ID,Operator (or Applicant),Site Name,Technology Type,Installed Capacity (MWelec),Development Status,Development Status (short),County,Region,Country,Post Code,X-coordinate,Y-coordinate,Planning Authority,Under Construction,Operational,Solar Site Area (sqm)
0,1,RWE npower,Aberthaw Power Station Biomass,Biomass (co-firing),35.00,Decommissioned,Decommissioned,South Glamorgan,Wales,Wales,CF62 4ZW,302219,166320,Vale of Glamorgan,01/05/2006,01/05/2007,NaN
1,2,Orsted (formerly Dong Energy) / Peel Energy,Hunterston - cofiring,Biomass (co-firing),170.00,Planning Application Withdrawn,Application Withdrawn,Strathclyde,Scotland,Scotland,NaN,218740,652937,Scottish Government (S36),NaN,NaN,NaN
2,3,Scottish and Southern Energy (SSE),Ferrybridge Multifuel 2 (FM2),EfW Incineration,70.00,Operational,Operational,West Yorkshire,Yorkshire and Humber,England,NaN,447490,424684,The Planning Inspectorate - National Infrastru...,01/09/2016,20/12/2019,NaN
3,4,Energy Power Resources,Thetford Biomass Power Station,Biomass (dedicated),38.50,Operational,Operational,Norfolk,Eastern,England,NaN,585300,286900,Breckland,NaN,02/10/1998,NaN
4,5,Agrigen,Nunn Mills Road Biomass Plant,Biomass (dedicated),8.80,Planning Permission Refused,Application Refused,Northamptonshire,East Midlands,England,NaN,476500,259700,West Northamptonshire,NaN,NaN,NaN


## Cleaning the development status field

Before filtering the dataset, I inspect and clean the `Development Status` field.

Real-world datasets often contain inconsistent spelling, capitalisation or hidden whitespace. For example, values such as `Operational`, `Operational ` and `operational` may look similar to a person, but Python treats them as different categories unless they are standardised.

To avoid accidentally excluding valid operational assets, I clean the status field by:

1. converting values to text,
2. removing leading and trailing spaces,
3. converting all values to lowercase.

Only after this cleaning step do I filter the dataset to operational assets.

In [3]:
assets["development_status_clean"] = (
    assets["Development Status"]
    .astype(str)
    .str.strip()
    .str.lower()
)

assets["development_status_clean"].value_counts()

development_status_clean
planning permission granted       5111
operational                       3060
planning application submitted    1745
planning permission refused        870
revised                            803
planning application withdrawn     637
under construction                 418
abandoned                          386
planning permission expired        336
appeal refused                     315
appeal granted                     190
appeal withdrawn                    49
decommissioned                      32
secretary of state - refusal        21
no application required             14
secretary of state - granted         5
appeal lodged                        3
Name: count, dtype: int64

In [4]:
operational_assets = assets[
    assets["development_status_clean"] == "operational"
].copy()

print("Operational assets:", len(operational_assets))

operational_assets.head()

Operational assets: 3060


,Ref ID,Operator (or Applicant),Site Name,Technology Type,Installed Capacity (MWelec),Development Status,Development Status (short),County,Region,Country,Post Code,X-coordinate,Y-coordinate,Planning Authority,Under Construction,Operational,Solar Site Area (sqm),development_status_clean
2,3,Scottish and Southern Energy (SSE),Ferrybridge Multifuel 2 (FM2),EfW Incineration,70.00,Operational,Operational,West Yorkshire,Yorkshire and Humber,England,NaN,447490,424684,The Planning Inspectorate - National Infrastru...,01/09/2016,20/12/2019,NaN,operational
3,4,Energy Power Resources,Thetford Biomass Power Station,Biomass (dedicated),38.50,Operational,Operational,Norfolk,Eastern,England,NaN,585300,286900,Breckland,NaN,02/10/1998,NaN,operational
13,14,Dalkia,Chilton Energy Plant,Biomass (dedicated),18.00,Operational,Operational,County Durham,North East,England,NaN,428049,530414,County Durham,01/03/2010,12/03/2012,NaN,operational
16,22,Double H Nurseries,Double H Nurseries Biomass Plant,Biomass (dedicated),1.50,Operational,Operational,Hampshire,South East,England,BH25 5NQ,422804,94625,New Forest,NaN,15/12/2012,NaN,operational
17,23,REACT Energy (Kedco),Newry Biomass Phase 1 (Gasification),Advanced Conversion Technologies,2.00,Operational,Operational,Co. Down,Northern Ireland,Northern Ireland,NaN,117392,488885,"Newry, Mourne and Down",NaN,27/06/2012,NaN,operational


## Checking coordinate completeness

Geospatial exposure analysis depends on asset location.

If an asset does not have an `X-coordinate` and `Y-coordinate`, it cannot be converted into a map point and cannot be used in distance-based exposure calculations.

For this project, operational assets without valid coordinates are removed from the geospatial modelling dataset.

In [5]:
coordinate_check = operational_assets[["X-coordinate", "Y-coordinate"]].isna().sum()

coordinate_check

X-coordinate    2
Y-coordinate    2
dtype: int64

In [6]:
operational_assets_with_coords = operational_assets.dropna(
    subset=["X-coordinate", "Y-coordinate"]
).copy()

print("Operational assets before coordinate filter:", len(operational_assets))
print("Operational assets with coordinates:", len(operational_assets_with_coords))
print(
    "Dropped due to missing coordinates:",
    len(operational_assets) - len(operational_assets_with_coords)
)

operational_assets_with_coords.head()

Operational assets before coordinate filter: 3060
Operational assets with coordinates: 3058
Dropped due to missing coordinates: 2


,Ref ID,Operator (or Applicant),Site Name,Technology Type,Installed Capacity (MWelec),Development Status,Development Status (short),County,Region,Country,Post Code,X-coordinate,Y-coordinate,Planning Authority,Under Construction,Operational,Solar Site Area (sqm),development_status_clean
2,3,Scottish and Southern Energy (SSE),Ferrybridge Multifuel 2 (FM2),EfW Incineration,70.00,Operational,Operational,West Yorkshire,Yorkshire and Humber,England,NaN,447490,424684,The Planning Inspectorate - National Infrastru...,01/09/2016,20/12/2019,NaN,operational
3,4,Energy Power Resources,Thetford Biomass Power Station,Biomass (dedicated),38.50,Operational,Operational,Norfolk,Eastern,England,NaN,585300,286900,Breckland,NaN,02/10/1998,NaN,operational
13,14,Dalkia,Chilton Energy Plant,Biomass (dedicated),18.00,Operational,Operational,County Durham,North East,England,NaN,428049,530414,County Durham,01/03/2010,12/03/2012,NaN,operational
16,22,Double H Nurseries,Double H Nurseries Biomass Plant,Biomass (dedicated),1.50,Operational,Operational,Hampshire,South East,England,BH25 5NQ,422804,94625,New Forest,NaN,15/12/2012,NaN,operational
17,23,REACT Energy (Kedco),Newry Biomass Phase 1 (Gasification),Advanced Conversion Technologies,2.00,Operational,Operational,Co. Down,Northern Ireland,Northern Ireland,NaN,117392,488885,"Newry, Mourne and Down",NaN,27/06/2012,NaN,operational


## Cleaning result

After filtering to operational assets, the dataset contains 3,060 records.

Two operational assets are missing coordinate values. These records are removed because they cannot be converted into map points or used in geospatial exposure analysis.

The working dataset therefore contains 3,058 operational renewable energy assets with valid coordinate data.

This cleaned dataset defines the asset universe for the first version of the project.

In [7]:
cleaned_csv_path = DATA_INTERIM / "operational_energy_assets_clean.csv"

operational_assets_with_coords.to_csv(cleaned_csv_path, index=False)

print("Saved cleaned asset table to:")
print(cleaned_csv_path)

Saved cleaned asset table to:
/Users/oconnor/jaurice-market-notes/projects/physical-climate-risk-uk-energy/data/interim/operational_energy_assets_clean.csv
